# Create Test Cases: Practice Exercise

Practice writing test cases for AI agent tools using the Arrange-Act-Assert pattern you learned in the guided practice.

**What you'll implement:**
- A test case for a string formatting tool
- A test case verifying agent tool selection
- A comprehensive test checking tool usage, parameters, and output

**Estimated time:** 10-15 minutes

In [ ]:
!pip install langchain-core langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 kB 16.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.19
    Uninstalling langchain-core-1.2.19:
      Successfully uninstalled langchain-core-1.2.19


In [ ]:

import os
from dotenv import load_dotenv
from typing import Dict, Any
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

def load_api_key():
  IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ


  if not IN_COLAB:
    from dotenv import load_dotenv
    load_dotenv()
    # Verify OpenAI API key is set
    if not os.getenv("OPENAI_API_KEY"):
      ("WARNING: OPENAI_API_KEY environment variable not set!")
  else:
    from google.colab import userdata
    OPENAI_API_KEY=userdata.get('OPEN_AI_API_KEY')
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY



## Setup

Run this cell to set up the tools and agent you'll be testing.

In [ ]:
load_api_key()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables")

In [ ]:
# Setup - run this cell first



# Tool 1: String Formatter
@tool
def string_formatter(text: str, format_type: str) -> str:
    """
    Formats a string according to the specified format type.

    Args:
        text: The input string to format
        format_type: The format to apply (uppercase, lowercase, title, reverse)

    Returns:
        The formatted string
    """
    format_functions = {
        "uppercase": lambda s: s.upper(),
        "lowercase": lambda s: s.lower(),
        "title": lambda s: s.title(),
        "reverse": lambda s: s[::-1]
    }

    if format_type not in format_functions:
        return f"Error: Unknown format type '{format_type}'"

    return format_functions[format_type](text)


# Tool 2: Word Counter
@tool
def word_counter(text: str, count_type: str) -> Dict[str, Any]:
    """
    Counts words or characters in a string.

    Args:
        text: The input string to analyze
        count_type: What to count (words, characters, characters_no_spaces)

    Returns:
        A dictionary with the count result and metadata
    """
    if count_type == "words":
        count = len(text.split())
    elif count_type == "characters":
        count = len(text)
    elif count_type == "characters_no_spaces":
        count = len(text.replace(" ", ""))
    else:
        return {"error": f"Unknown count type: {count_type}"}

    return {
        "original_text": text,
        "count_type": count_type,
        "count": count
    }


# Create agent with tools
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
tools = [string_formatter, word_counter]
agent = create_agent(
    model,
    tools=tools,
    system_prompt="You are a helpful assistant with access to text processing tools. Use them when needed."
)

print("Setup complete!")
print(f"Available tools: {[t.name for t in tools]}")

Setup complete!
Available tools: ['string_formatter', 'word_counter']


## Context

You have access to two text processing tools:

1. **string_formatter**: Takes `text` and `format_type` (uppercase, lowercase, title, reverse). Returns the formatted string.

2. **word_counter**: Takes `text` and `count_type` (words, characters, characters_no_spaces). Returns a dictionary with `original_text`, `count_type`, and `count`.

Your task is to write test cases that verify:
- Tools work correctly in isolation
- The agent selects the right tool for a task
- The agent extracts correct parameters and provides accurate output

Use the **Arrange-Act-Assert** pattern for all tests.

## Task 1: Test the string_formatter Tool

Write a test that verifies `string_formatter` correctly converts "hello world" to title case ("Hello World").

In [ ]:
def test_string_formatter_title_case():
    """
    Test that string_formatter correctly converts text to title case.

    Input: text="hello world", format_type="title"
    Expected output: "Hello World"

    Follow the Arrange-Act-Assert pattern.
    """
    # TODO: Implement the test
    # ARRANGE: Set up test inputs (text, format_type, expected_result)

    # ACT: Invoke the tool using string_formatter.invoke({...})

    # ASSERT: Verify result equals expected_result
    text = "hello world"
    format_type = "title"
    expected_result = "Hello World"
    result = string_formatter.invoke({"text": text, "format_type": format_type})
    assert result == expected_result


# Run your test
test_string_formatter_title_case()

## Task 2: Test Agent Tool Selection

Write a test that verifies the agent uses `word_counter` when asked to count words in a sentence.

In [ ]:
def test_agent_uses_word_counter():
    """
    Test that the agent selects word_counter when asked to count words.

    Query: "How many words are in the sentence: The quick brown fox jumps"
    Expected tool: "word_counter"

    Steps:
    1. Invoke the agent with the query
    2. Extract tool calls from messages
    3. Assert that "word_counter" was called
    """
    # TODO: Implement the test
    # ARRANGE: Set up query and expected_tool

    # ACT: Invoke agent with agent.invoke({"messages": [{"role": "user", "content": query}]})

    # ASSERT: Extract tool_calls from messages and verify word_counter was used
    # Hint: Loop through messages, check for tool_calls attribute, collect tool names


    query = "How many words are in the sentence: The quick brown fox jumps"
    expected_tool = "word_counter"
    messages = [{"role": "user", "content": query}]
    result = agent.invoke({"messages": messages})
    result_messages = result.get("messages",[])
    tool_calls = []
    for message in result_messages:
      if hasattr(message, "tool_calls") and message.tool_calls:
        for tool_call in message.tool_calls:
          tool_calls.append(tool_call.get("name"))

    assert expected_tool in tool_calls, f"Expected tool {expected_tool} not found in {tool_calls}"

# Run your test
test_agent_uses_word_counter()

## Task 3: Test Agent Parameter Extraction

Write a test that verifies the agent correctly extracts parameters from a query and passes them to the tool.

In [ ]:
def test_agent_parameter_extraction():
    """
    Test that the agent correctly extracts parameters from a query.

    Query: "Format 'hello world' as uppercase"
    Expected tool: "string_formatter"
    Expected parameters: text="hello world", format_type="uppercase"

    Steps:
    1. Invoke the agent with the query
    2. Find the tool call for string_formatter
    3. Verify the tool_input contains correct text and format_type
    """
    # TODO: Implement the test
    # ARRANGE: Set up query, expected_tool, expected_text, expected_format_type

    # ACT: Invoke the agent

    # ASSERT: Find tool calls, verify string_formatter was called with correct parameters
    # Hint: Loop through messages, find tool_calls, check tool_call["args"] for parameters

    query ="Format 'hello world' as uppercase"
    expected_tool = "string_formatter"
    expected_text, expected_format_type = "hello world", "uppercase"
    messages = [{"role": "user", "content": query}]
    result = agent.invoke({"messages": messages})
    result_messages = result.get("messages",[])
    tool_input = None
    for message in result_messages:
      if hasattr(message, "tool_calls") and message.tool_calls:
        for tool_call in message.tool_calls:
          if tool_call.get("name") == expected_tool:
            tool_name = tool_call.get("name")
            tool_input = tool_call.get("args")
    assert tool_name == expected_tool, f"Expected tool {expected_tool} not found"
    assert expected_text == tool_input["text"], f"Expected text {expected_text}, but got {tool_input[0]}"
    assert expected_format_type == tool_input["format_type"], f"Expected format_type {expected_format_type}, but got {tool_input[1]}"
# Run your test
test_agent_parameter_extraction()

## Run Your Implementation

Execute all your tests to verify they work correctly.

In [ ]:
# Run all tests
print("Running all tests...\n")

print("Test 1: String Formatter Tool")
test_string_formatter_title_case()
print()

print("Test 2: Agent Tool Selection")
test_agent_uses_word_counter()
print()

print("Test 3: Agent Parameter Extraction")
test_agent_parameter_extraction()
print()

print("All tests completed!")

Running all tests...

Test 1: String Formatter Tool

Test 2: Agent Tool Selection

Test 3: Agent Parameter Extraction

All tests completed!
